In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ville").getOrCreate()
"""Charger le fichier ville_1.csv dans un DataFrame et afficher les premières lignes"""
df = spark.read.csv( r"/content/sample_data/ville_1.csv",header=True, inferSchema=True)
#df.show(10)
"""Afficher les types de colonnes de ce fichier"""
df.dtypes
nbre_ligne_fichier_ville=df.count()
print(nbre_ligne_fichier_ville)

1083


In [2]:
df_homme=df.filter(df.sexe=="H")
df_femme=df.filter(df.sexe=="F")

In [3]:
df.groupBy("sexe").count().show()

df.createOrReplaceTempView("villes")
spark.sql("SELECT COUNT(*), SEXE  AS nb FROM VILLES GROUP BY SEXE ").show()

+----+-----+
|sexe|count|
+----+-----+
|   F|  523|
|   H|  560|
+----+-----+

+--------+---+
|count(1)| nb|
+--------+---+
|     523|  F|
|     560|  H|
+--------+---+



In [4]:
spark.sql("SELECT AVG(SALAIRE) AS MOYENNE_SALAIRE FROM VILLES ").show()

+------------------+
|   MOYENNE_SALAIRE|
+------------------+
|25943.173561933854|
+------------------+



In [8]:
from pyspark.sql.functions import avg,mean
df.agg(avg("salaire")).show()
df.show()
from pyspark.sql.types import DoubleType
df=df.withColumn("salary",df["salaire"].cast(DoubleType()))
moyenne=df.select(mean("salary")).show()



+------------------+
|      avg(salaire)|
+------------------+
|25943.173561933854|
+------------------+

+----+--------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+
|  id|      vitesse_a_pied|     vitesse_a_velo|                home|             travail|sportif|casseur|              statut|           salaire|sexe|age|         sportivite|velo_perf_minimale|            salary|
+----+--------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+
|5251|                0.02|               0.05|(lon:26.60 lat:28...|(lon:21.08 lat:14...|  false|  false|          reserviste|29800.610034665042|   F| 18|                0.1|               0.4|29800.610034665042|
|5252| 0.14974625830876215|0.3743656457719

In [9]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def tranche(salaire):
    if salaire is None:
        return None
    if salaire < 10000:
        return 0
    return int(salaire // 10000)

classe = udf(tranche, IntegerType())
df = df.withColumn("Classe", classe(df["salaire"]))
df.show()

+----+--------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+------+
|  id|      vitesse_a_pied|     vitesse_a_velo|                home|             travail|sportif|casseur|              statut|           salaire|sexe|age|         sportivite|velo_perf_minimale|            salary|Classe|
+----+--------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+------+
|5251|                0.02|               0.05|(lon:26.60 lat:28...|(lon:21.08 lat:14...|  false|  false|          reserviste|29800.610034665042|   F| 18|                0.1|               0.4|29800.610034665042|     2|
|5252| 0.14974625830876215|0.37436564577190534|(lon:0.26 lat:42.61)|(lon:36.35 lat:33...|  false|  false|          profe

In [10]:
tranche_salaire=df.groupBy("Classe").count()
tranche_salaire.show()

+------+-----+
|Classe|count|
+------+-----+
|     1|  295|
|     6|    3|
|     3|  222|
|     5|   14|
|     4|   65|
|     8|    1|
|     7|    1|
|     2|  476|
|     0|    6|
+------+-----+



In [11]:
df.sort("sexe","classe").show()

+----+-------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+------+
|  id|     vitesse_a_pied|     vitesse_a_velo|                home|             travail|sportif|casseur|              statut|           salaire|sexe|age|         sportivite|velo_perf_minimale|            salary|Classe|
+----+-------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+------------------+------+
|5375|               0.02|               0.05|(lon:26.11 lat:3.34)|(lon:25.13 lat:7.51)|  false|  false|             employe|  7392.30996558039|   F| 55|                0.1|               0.4|  7392.30996558039|     0|
|6031|0.08328838658186277|0.20822096645465696|(lon:19.32 lat:8.60)|(lon:37.95 lat:25...|  false|  false|technicien_de_sur...

In [18]:
from pyspark.sql.functions import collect_list,collect_set,size
df.groupBy("sexe").agg(size(collect_set("id")),size(collect_set("classe"))).show()


+----+---------------------+-------------------------+
|sexe|size(collect_set(id))|size(collect_set(classe))|
+----+---------------------+-------------------------+
|   F|                  523|                        7|
|   H|                  560|                        9|
+----+---------------------+-------------------------+

